# Day 13 練習：50 字摘要會帶走什麼？

## 目標

把一個約 300 字的故事交給 Qwen 壓成 50 字，再比較摘要保留和遺失了哪些元素。這是一次**單一故事、單一模型、單次執行**的教學觀察，不是模型能力排名。

重點不是問「摘要順不順」，而是問：「若下一位 Agent 只看到這份摘要，它還能做對後續決定嗎？」

## 0. 執行前準備

請從 repository 根目錄或 `Day13` 資料夾啟動此 Notebook，並在專案根目錄的 `.env` 放入：

```text
GROQ_API_KEY=你的金鑰
```

這份練習會把下方的虛構故事傳給 Groq 上的 Qwen 模型。請勿把真實個資或機密文字貼進去。模型名稱集中在設定格，若你的帳戶沒有權限或模型已更新，只需改那一行。

若出現 `ModuleNotFoundError`，請在新的 cell 手動執行：`%pip install groq python-dotenv pandas`，再從第一個程式 cell 重新開始。

In [1]:
import os
from textwrap import dedent

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from groq import Groq
from IPython.display import Markdown, display

load_dotenv(find_dotenv(usecwd=True))

MODEL = "qwen/qwen3.6-27b"
MAX_SUMMARY_CHARS = 50

if not os.environ.get("GROQ_API_KEY"):
    raise RuntimeError(
        "找不到 GROQ_API_KEY。請在專案根目錄的 .env 設定金鑰後，再重新執行。"
    )

print(f"模型：{MODEL}；目標長度：不超過 {MAX_SUMMARY_CHARS} 個字元")

模型：qwen/qwen3.6-27b；目標長度：不超過 50 個字元


## 1. 固定一則故事，也固定我們在乎的元素

先把故事與檢查點寫死，避免模型輸出後才倒過來決定「什麼算重要」。以下元素並非唯一正解；它們只是模擬後續工作可能需要的事實。

In [3]:
story = dedent("""
星期六下午四點，國三生張阿明牽著剛修好的藍色腳踏車，準備趕到圖書館參加志工面談。車籃裡還放著要帶回家的紅豆餅，他也答應媽媽六點前回家餵那隻怕雷聲的橘貓。
天空突然下起大雨，他躲進公車站時，看見長椅下有一個深綠色皮夾。裡面除了身分證和現金，
還夾著一張寫著「給爸爸的生日卡」的手寫便條。阿明原本想直接交給派出所，卻發現失主陳伯伯
的住址就在附近，而且便條寫著今晚七點要在醫院替爸爸慶生。他先打電話給圖書館說會遲到，
再依身分證地址找去。陳伯伯正焦急地在巷口找皮夾；他收到後連聲道謝，說卡片是孫女第一次
親手寫給他的。阿明最後錯過面談，卻收到館員訊息：面談改到明天早上十點。
""").strip()

# 每列是一個可能影響後續判斷的元素；同一列可有多個可接受詞。
story_elements = [
    ("主角", "張阿明", ["張阿明", "阿明"]),
    ("原本目標", "去圖書館參加志工面談", ["圖書館", "志工", "面談"]),
    ("時間", "星期六下午四點", ["星期六", "下午四點", "四點"]),
    ("發現物品", "深綠色皮夾", ["皮夾", "錢包", "深綠"]),
    ("關鍵線索", "給爸爸的生日卡", ["生日卡", "生日", "卡片"]),
    ("時間壓力", "今晚七點在醫院慶生", ["七點", "醫院", "慶生"]),
    ("主角的取捨", "先通知圖書館，再送回皮夾", ["通知圖書館", "打電話", "送回", "歸還"]),
    ("失主的情感", "孫女第一次親手寫卡", ["孫女", "第一次", "親手"]),
    ("最後結果", "錯過面談，但改到明早十點", ["明天", "十點", "改", "錯過面談"]),
]

assert 250 <= len(story) <= 380, f"故事長度為 {len(story)}，請維持約 300 字。"
assert len(story_elements) == 9

display(Markdown(f"### 原始故事（{len(story)} 字元）\n\n{story}"))
pd.DataFrame(story_elements, columns=["類別", "原文中的重要元素", "自動檢查可接受詞"])[["類別", "原文中的重要元素"]]

### 原始故事（285 字元）

星期六下午四點，國三生張阿明牽著剛修好的藍色腳踏車，準備趕到圖書館參加志工面談。車籃裡還放著要帶回家的紅豆餅，他也答應媽媽六點前回家餵那隻怕雷聲的橘貓。
天空突然下起大雨，他躲進公車站時，看見長椅下有一個深綠色皮夾。裡面除了身分證和現金，
還夾著一張寫著「給爸爸的生日卡」的手寫便條。阿明原本想直接交給派出所，卻發現失主陳伯伯
的住址就在附近，而且便條寫著今晚七點要在醫院替爸爸慶生。他先打電話給圖書館說會遲到，
再依身分證地址找去。陳伯伯正焦急地在巷口找皮夾；他收到後連聲道謝，說卡片是孫女第一次
親手寫給他的。阿明最後錯過面談，卻收到館員訊息：面談改到明天早上十點。

,類別,原文中的重要元素
0,主角,張阿明
1,原本目標,去圖書館參加志工面談
2,時間,星期六下午四點
3,發現物品,深綠色皮夾
4,關鍵線索,給爸爸的生日卡
5,時間壓力,今晚七點在醫院慶生
6,主角的取捨,先通知圖書館，再送回皮夾
7,失主的情感,孫女第一次親手寫卡
8,最後結果,錯過面談，但改到明早十點


## 2. 請 Qwen 壓成 50 字

先看原始模型輸出，再做任何自動檢查。每次執行可能略有不同；這正好可以讓你比較不同摘要遺失的元素是否相同。

In [4]:
client = Groq(api_key=os.environ["GROQ_API_KEY"])

prompt = f"""請將下面故事摘要為不超過 {MAX_SUMMARY_CHARS} 個中文字元。
只輸出摘要本身，不要標題、條列、說明或引號。

故事：
{story}
"""

response = client.chat.completions.create(
    model=MODEL,
    reasoning_effort="none",
    temperature=0,
    max_completion_tokens=120,
    messages=[{"role": "user", "content": prompt}],
)

if not response.choices:
    raise RuntimeError("模型沒有回傳 choices，無法進行比較。")
if response.choices[0].finish_reason == "length":
    raise RuntimeError("回應被 token 上限截斷；請提高 max_completion_tokens 後重試。")

summary = (response.choices[0].message.content or "").strip()
if not summary:
    raise RuntimeError("模型回傳空白摘要，無法進行比較。")

display(Markdown("### Qwen 的原始摘要\n\n> " + summary.replace("\n", " ") + f"\n\n字元數：**{len(summary)}**"))
if len(summary) > MAX_SUMMARY_CHARS:
    print(f"提醒：模型輸出 {len(summary)} 字元，超過目標 {MAX_SUMMARY_CHARS} 字元。這也是一種可觀察的失敗。")

### Qwen 的原始摘要

> 阿明冒雨還失物雖錯過志工面談，卻獲改期，並助陳伯伯趕上父親生日。

字元數：**32**

## 3. 對照：哪些元素還看得到？

下面的「命中」只是在摘要裡找到預先設定的詞，**不是摘要品質分數，也不能證明模型理解了故事**。例如模型可能用同義詞說對了，卻被判成未命中；也可能提到一個詞，卻誤解它的意義。請把它當作提醒你再回頭讀原文的螢光筆。

In [5]:
audit_rows = []
for category, element, accepted_terms in story_elements:
    hits = [term for term in accepted_terms if term in summary]
    audit_rows.append({
        "類別": category,
        "原文的重要元素": element,
        "自動線索": "可能保留" if hits else "可能遺失",
        "摘要中找到的詞": "、".join(hits) or "—",
    })

audit = pd.DataFrame(audit_rows)
display(audit)

possible_loss = (audit["自動線索"] == "可能遺失").sum()
print(f"自動線索：9 個預設元素中，有 {possible_loss} 個未找到預設詞。請人工確認，不要把這個數字當成模型分數。")

,類別,原文的重要元素,自動線索,摘要中找到的詞
0,主角,張阿明,可能保留,阿明
1,原本目標,去圖書館參加志工面談,可能保留,志工、面談
2,時間,星期六下午四點,可能遺失,—
3,發現物品,深綠色皮夾,可能遺失,—
4,關鍵線索,給爸爸的生日卡,可能保留,生日
5,時間壓力,今晚七點在醫院慶生,可能遺失,—
6,主角的取捨,先通知圖書館，再送回皮夾,可能遺失,—
7,失主的情感,孫女第一次親手寫卡,可能遺失,—
8,最後結果,錯過面談，但改到明早十點,可能保留,改


自動線索：9 個預設元素中，有 5 個未找到預設詞。請人工確認，不要把這個數字當成模型分數。


## 4. 請用人的眼睛檢查

請先自己回答，再和同學比較。沒有唯一答案；重點是你能指出「少了什麼，因此哪個後續決定可能做錯」。

1. 摘要保留了事件主線嗎？
2. 哪一個遺失元素會改變你對以安行為的理解？為什麼？
3. 如果下一位 Agent 要判斷「面談是否還要重約」，它只看 50 字摘要夠嗎？
4. 若任務目標改成「找出皮夾失主的急迫性」，你會要求摘要保留哪幾項？
5. 請把提示改成「優先保留時間、地點與後續待辦」，再跑一次。這次又遺失了什麼？

> 摘要不是壞東西；它是有取捨的壓縮。真正要問的是：它留下的資訊，還夠不夠支援下一步？

讓AI 做 summary是大多數人很常使用的方式，但鮮少人去討論「摘要留下了什麼、遺失了什麼」。 試著去思考看看，如果你是針對一場會議錄音，並將會議作成摘要，什麼元素會被保留？什麼元素會失去？這些遺失的元素會不會影響到後續的決策或行動？ 所以回過頭來，我們還是要知道自己真實需要的東西是什麼，才可以藉由prompt去引導AI，讓它幫我們做出最符合需求的摘要。

## 延伸：這和 Compaction 有什麼關係？

Compaction 也是把很長的歷史壓成較短版本，好讓下一輪繼續工作。這個小實驗故意把壓縮比例拉得很大，讓遺失變得容易看見。真實工作中，好的 Compaction 會依下一步任務保留目標、已決定事項、失敗嘗試、未解問題與關鍵證據，而不只是寫一段讀起來漂亮的摘要。